In [1]:
# Run this only if NumPy or PyTorch is not installed
%pip install -q numpy torch

Note: you may need to restart the kernel to use updated packages.


In [2]:
import math
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

from multiprocessing import Pool

# Reproducibility
SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

print("PyTorch version:", torch.__version__)

PyTorch version: 2.14.0+cpu


In [3]:
# ---------- 1. Game ----------

LINES = [
    (0, 1, 2),
    (3, 4, 5),
    (6, 7, 8),
    (0, 3, 6),
    (1, 4, 7),
    (2, 5, 8),
    (0, 4, 8),
    (2, 4, 6),
]


def moves(s):
    """Return all legal moves."""
    return [i for i in range(9) if s[i] == 0]


def play(s, a):
    """
    Play action a and flip the board perspective.

    1  = current player's pieces
    -1 = opponent's pieces
    0  = empty square
    """
    s = s.copy()
    s[a] = 1
    return -s


def result(s):
    """
    Return the value for the player to move.

    -1.0 : opponent just won
     0.0 : draw
    None : game is still ongoing
    """
    for a, b, c in LINES:
        if s[a] == s[b] == s[c] == -1:
            return -1.0

    if 0 not in s:
        return 0.0

    return None


# Quick test
initial_state = np.zeros(9, dtype=np.float32)
print("Legal moves:", moves(initial_state))
print("Initial result:", result(initial_state))

Legal moves: [0, 1, 2, 3, 4, 5, 6, 7, 8]
Initial result: None


In [4]:
# ---------- 2. Network ----------

class Net(nn.Module):
    def __init__(self):
        super().__init__()

        self.body = nn.Sequential(
            nn.Linear(9, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU()
        )

        self.pi = nn.Linear(64, 9)
        self.v = nn.Linear(64, 1)

    def forward(self, x):
        h = self.body(x)

        # Policy logits and scalar value
        policy_logits = self.pi(h)
        value = torch.tanh(self.v(h)).squeeze(-1)

        return policy_logits, value


def predict(net, s):
    """
    Predict a legal policy distribution and value for state s.
    """
    net.eval()

    with torch.no_grad():
        x = torch.tensor(
            s,
            dtype=torch.float32
        ).unsqueeze(0)

        logits, value = net(x)

        # Convert logits to probabilities
        p = F.softmax(logits, dim=1)[0].cpu().numpy()

    # Mask illegal moves
    mask = np.zeros(9, dtype=np.float32)
    mask[moves(s)] = 1.0

    p = p * mask

    # Normalize over legal moves only
    p = p / p.sum()

    return p, float(value.item())


# Test the network
test_net = Net()
test_policy, test_value = predict(test_net, initial_state)

print("Policy:", test_policy)
print("Policy sum:", test_policy.sum())
print("Value:", test_value)

Policy: [0.12160758 0.11319287 0.10923736 0.10433909 0.10527755 0.11694823
 0.10104786 0.1164595  0.11189002]
Policy sum: 1.0
Value: -0.01783614233136177


In [5]:
# ---------- 3. MCTS ----------

class Node:
    def __init__(self, p):
        self.p = float(p)   # Prior probability
        self.n = 0           # Visit count
        self.w = 0.0         # Total value
        self.kids = {}       # Child nodes


def expand(net, node, s):
    """
    Expand a node using the neural network policy.
    Return the neural network value estimate.
    """
    p, v = predict(net, s)

    for a in moves(s):
        node.kids[a] = Node(p[a])

    return v


def mcts(net, s0, sims=50, c=1.5):
    """
    Run Monte Carlo Tree Search from state s0.

    Returns:
        Improved policy based on visit counts.
    """
    root = Node(1.0)

    # Expand root
    expand(net, root, s0)

    for _ in range(sims):
        node = root
        s = s0
        path = [root]

        # ---------- Selection ----------
        while node.kids:
            parent_visits = math.sqrt(node.n + 1)

            def puct_score(item):
                action, child = item

                # Q-value from the current player's perspective
                if child.n:
                    q = -child.w / child.n
                else:
                    q = 0.0

                # Exploration bonus
                u = c * child.p * parent_visits / (1 + child.n)

                return q + u

            action, node = max(
                node.kids.items(),
                key=puct_score
            )

            # Apply action and switch perspective
            s = play(s, action)
            path.append(node)

        # ---------- Evaluation ----------
        v = result(s)

        if v is None:
            # Expand and evaluate with the network
            v = expand(net, node, s)

        # ---------- Backup ----------
        # Flip the value at every level because the player changes
        for n in reversed(path):
            n.n += 1
            n.w += v
            v = -v

    # Improved policy from visit counts
    pi = np.zeros(9, dtype=np.float32)

    for action, child in root.kids.items():
        pi[action] = child.n

    pi /= pi.sum()

    return pi


# Test MCTS
test_policy = mcts(test_net, initial_state, sims=50)

print("MCTS policy:", test_policy)
print("MCTS policy sum:", test_policy.sum())
print("Selected action:", np.argmax(test_policy))

MCTS policy: [0.1  0.12 0.1  0.1  0.1  0.14 0.1  0.12 0.12]
MCTS policy sum: 1.0
Selected action: 5


In [6]:
# ---------- 4. Self-play ----------

def self_play(net, sims=50):
    """
    Generate one self-play game.

    Returns a list of:
        (state, improved_policy, value_target)
    """
    s = np.zeros(9, dtype=np.float32)
    history = []

    while True:
        # Use MCTS to obtain an improved policy
        pi = mcts(net, s, sims=sims)

        # Save the current state and policy
        history.append((s.copy(), pi.copy()))

        # Sample an action from the MCTS policy
        action = np.random.choice(9, p=pi)

        # Play action and switch perspective
        s = play(s, action)

        # Check game result
        r = result(s)

        if r is not None:
            break

    # The result r is from the perspective of the player to move
    # after the final move. Convert it into targets for every state.
    training_data = []

    for i, (state, policy) in enumerate(reversed(history)):
        value_target = -r if i % 2 == 0 else r
        training_data.append(
            (state, policy, value_target)
        )

    return training_data


def worker(args):
    """
    Worker used by the optional multiprocessing version.
    """
    state_dict, n_games, worker_seed = args

    np.random.seed(worker_seed)
    torch.manual_seed(worker_seed)
    torch.set_num_threads(1)

    net = Net()
    net.load_state_dict(state_dict)

    data = []

    for _ in range(n_games):
        data.extend(self_play(net, sims=50))

    return data


# Test one self-play game
test_game = self_play(test_net, sims=50)

print("Examples generated:", len(test_game))
print("First example state:", test_game[0][0])
print("First example policy:", test_game[0][1])
print("First example value:", test_game[0][2])

Examples generated: 9
First example state: [ 1. -1.  0. -1.  1.  1. -1.  1. -1.]
First example policy: [0. 0. 1. 0. 0. 0. 0. 0. 0.]
First example value: -0.0


In [7]:
# ---------- 5. Training ----------

def train(net, data, steps=50, learning_rate=1e-3):
    """
    Train the network using self-play data.
    """
    if len(data) == 0:
        raise ValueError("Training data is empty.")

    net.train()

    states = np.asarray(
        [d[0] for d in data],
        dtype=np.float32
    )

    policies = np.asarray(
        [d[1] for d in data],
        dtype=np.float32
    )

    values = np.asarray(
        [d[2] for d in data],
        dtype=np.float32
    )

    S = torch.tensor(states)
    P = torch.tensor(policies)
    Z = torch.tensor(values)

    optimizer = torch.optim.Adam(
        net.parameters(),
        lr=learning_rate
    )

    for _ in range(steps):
        logits, values_pred = net(S)

        # Policy loss
        policy_loss = -(
            P * F.log_softmax(logits, dim=1)
        ).sum(dim=1).mean()

        # Value loss
        value_loss = F.mse_loss(values_pred, Z)

        # Combined loss
        loss = policy_loss + value_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    return float(loss.item())

In [8]:
# ---------- 6. Main training loop ----------
# Recommended notebook-safe version

net = Net()
buffer = []

N_ITERATIONS = 10
GAMES_PER_ITERATION = 32
MCTS_SIMS = 50
MAX_BUFFER_SIZE = 5000

for iteration in range(N_ITERATIONS):
    iteration_data = []

    # Generate 32 self-play games
    for _ in range(GAMES_PER_ITERATION):
        iteration_data.extend(
            self_play(net, sims=MCTS_SIMS)
        )

    # Add new examples to replay buffer
    buffer.extend(iteration_data)

    # Keep only recent examples
    buffer = buffer[-MAX_BUFFER_SIZE:]

    # Train on the current buffer
    loss = train(
        net,
        buffer,
        steps=50,
        learning_rate=1e-3
    )

    print(
        f"iter {iteration + 1}: "
        f"examples={len(buffer)}, "
        f"loss={loss:.3f}"
    )

iter 1: examples=243, loss=2.522
iter 2: examples=456, loss=2.437
iter 3: examples=676, loss=2.301
iter 4: examples=899, loss=2.104
iter 5: examples=1121, loss=1.984
iter 6: examples=1332, loss=1.870
iter 7: examples=1572, loss=1.761
iter 8: examples=1819, loss=1.681
iter 9: examples=2064, loss=1.609
iter 10: examples=2308, loss=1.555
